# Dataset Cleaning — Self-Healing MLOps Comparison Study

This notebook cleans and prepares the **3 datasets** used in the comparison study:

| Dataset | Source | Raw Rows | Raw Cols | After Cleaning |
|---------|--------|----------|----------|----------------|
| Credit Card Fraud | Kaggle | 284,807 | 31 | 29 features, target=Class |
| Elec2 | river library | 45,312 | 9 | 7 features, target=UP/DOWN |
| LendingClub | Kaggle | ~2.2M | 151 | 18 features, target=loan_status |

**Output:** Three clean CSV files ready for the harness:
- `fraud_clean.csv`
- `elec2_clean.csv`
- `lendingclub_clean.csv`

## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print(f"pandas {pd.__version__}, numpy {np.__version__}")

---
## 1. Credit Card Fraud Dataset

**Source:** [Kaggle — Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

**Raw columns:** `Time`, `V1`–`V28` (PCA-anonymised), `Amount`, `Class`

**Cleaning steps:**
1. Drop `Time` (seconds from first txn — not a real feature)
2. Keep `V1`–`V28` + `Amount` as features (29 total)
3. Rename `Class` → `target` (already 0/1)
4. Fill any NaN with 0
5. Verify extreme imbalance (~0.17% positive)

In [ ]:
# ── Upload or load the Kaggle CSV ──
# On Colab: upload creditcard.csv from Kaggle
# Locally: point to your file path

try:
    # Try Colab upload
    from google.colab import files
    print("Upload creditcard.csv from Kaggle:")
    uploaded = files.upload()
    fraud_path = list(uploaded.keys())[0]
except ImportError:
    # Local path
    fraud_path = r"C:\claude\fraud_clean.csv"  # Change this to your path

print(f"Reading from: {fraud_path}")

In [ ]:
# ── Load raw data ──
fraud_raw = pd.read_csv(fraud_path)
print("=" * 60)
print("FRAUD — RAW DATA")
print("=" * 60)
print(f"Shape: {fraud_raw.shape}")
print(f"Columns: {list(fraud_raw.columns)}")
print(f"\nDtypes:\n{fraud_raw.dtypes.value_counts()}")
print(f"\nNaN per column:\n{fraud_raw.isna().sum()[fraud_raw.isna().sum() > 0]}")
if fraud_raw.isna().sum().sum() == 0:
    print("  (no NaN)")
print(f"\nTarget column distribution:")

# Detect target column name
if 'Class' in fraud_raw.columns:
    target_col = 'Class'
elif 'target' in fraud_raw.columns:
    target_col = 'target'
else:
    raise KeyError(f"No target column found. Columns: {list(fraud_raw.columns)}")

print(fraud_raw[target_col].value_counts())
print(f"Positive rate: {fraud_raw[target_col].mean():.4f}")

In [ ]:
# ── Clean fraud dataset ──
print("Cleaning fraud dataset...")

# Step 1: Select features (drop Time, keep V1-V28 + Amount)
feature_cols = [c for c in fraud_raw.columns if c.startswith('V')] + ['Amount']
fraud_clean = fraud_raw[feature_cols].copy()

# Step 2: Rename target
fraud_clean['target'] = fraud_raw[target_col].astype(int)

# Step 3: Convert to numeric, fill NaN
for col in feature_cols:
    fraud_clean[col] = pd.to_numeric(fraud_clean[col], errors='coerce')
fraud_clean[feature_cols] = fraud_clean[feature_cols].fillna(0.0)

# Step 4: Verify
print("\n" + "=" * 60)
print("FRAUD — CLEANED")
print("=" * 60)
print(f"Shape: {fraud_clean.shape}")
print(f"Features ({len(feature_cols)}): {feature_cols[:5]}... + {feature_cols[-1]}")
print(f"NaN cells: {fraud_clean.isna().sum().sum()}")
print(f"Target: {{0: {(fraud_clean['target']==0).sum():,}, 1: {(fraud_clean['target']==1).sum():,}}}")
print(f"Positive rate: {fraud_clean['target'].mean():.4f} ({fraud_clean['target'].mean()*100:.2f}%)")
print(f"\nAll dtypes numeric: {all(fraud_clean[feature_cols].dtypes.apply(lambda x: np.issubdtype(x, np.number)))}")
print("\n✅ Fraud dataset cleaned successfully")

In [ ]:
# ── Save cleaned fraud ──
fraud_clean.to_csv('fraud_clean.csv', index=False)
print(f"Saved: fraud_clean.csv ({fraud_clean.shape[0]:,} rows, {fraud_clean.shape[1]} cols)")

---
## 2. Elec2 Dataset

**Source:** `river` Python library (`river.datasets.Elec2`)

**Raw columns:** `date`, `day`, `period`, `nswprice`, `nswdemand`, `vicprice`, `vicdemand`, `transfer`, `class`

**Cleaning steps:**
1. Generate from `river` library (stream → DataFrame)
2. Keep 7 numeric features: `date`, `day`, `period`, `nswprice`, `nswdemand`, `vicprice`, `vicdemand`
3. Encode target: `UP` → 1, `DOWN` → 0
4. Fill NaN with median
5. **NEVER shuffle** — row order IS the temporal drift signal

> ⚠️ **Critical:** This dataset has REAL temporal drift around row 17,424. 
> The order of rows must be preserved exactly.

In [ ]:
# ── Install river if needed ──
try:
    from river.datasets import Elec2
    print("river library found")
except ImportError:
    print("Installing river...")
    !pip install river -q
    from river.datasets import Elec2
    print("river installed")

In [ ]:
# ── Generate Elec2 from river ──
print("Generating Elec2 from river library...")
dataset = Elec2()

rows = []
targets = []
for x, y_val in dataset:
    rows.append(x)
    targets.append(y_val)

elec2_raw = pd.DataFrame(rows)
elec2_raw['class'] = targets

print("\n" + "=" * 60)
print("ELEC2 — RAW DATA")
print("=" * 60)
print(f"Shape: {elec2_raw.shape}")
print(f"Columns: {list(elec2_raw.columns)}")
print(f"\nDtypes:\n{elec2_raw.dtypes}")
print(f"\nTarget distribution:\n{elec2_raw['class'].value_counts()}")
print(f"\nNaN per column:\n{elec2_raw.isna().sum()}")

In [ ]:
# ── Clean elec2 dataset ──
print("Cleaning Elec2 dataset...")

elec2_clean = elec2_raw.copy()

# Step 1: Encode target (True/False or UP/DOWN → 1/0)
target_series = elec2_clean['class']
if target_series.dtype == bool:
    elec2_clean['target'] = target_series.astype(int)
elif target_series.dtype == object:
    mapping = {'UP': 1, 'DOWN': 0, 'True': 1, 'False': 0}
    elec2_clean['target'] = target_series.map(mapping).astype(int)
else:
    elec2_clean['target'] = target_series.astype(int)

# Step 2: Drop original class column
elec2_clean = elec2_clean.drop(columns=['class'])

# Step 3: Identify feature columns (everything except target)
feature_cols_e = [c for c in elec2_clean.columns if c != 'target']

# Step 4: Convert to numeric, fill NaN with median
for col in feature_cols_e:
    elec2_clean[col] = pd.to_numeric(elec2_clean[col], errors='coerce')
elec2_clean[feature_cols_e] = elec2_clean[feature_cols_e].fillna(
    elec2_clean[feature_cols_e].median()
)

# Step 5: DO NOT SHUFFLE — temporal order is the drift signal
# (rows are already in chronological order from river)

# Verify
print("\n" + "=" * 60)
print("ELEC2 — CLEANED")
print("=" * 60)
print(f"Shape: {elec2_clean.shape}")
print(f"Features ({len(feature_cols_e)}): {feature_cols_e}")
print(f"NaN cells: {elec2_clean.isna().sum().sum()}")
print(f"Target: {{0: {(elec2_clean['target']==0).sum():,}, 1: {(elec2_clean['target']==1).sum():,}}}")
print(f"Positive rate: {elec2_clean['target'].mean():.4f} ({elec2_clean['target'].mean()*100:.2f}%)")
print(f"All dtypes numeric: {all(elec2_clean[feature_cols_e].dtypes.apply(lambda x: np.issubdtype(x, np.number)))}")
print("\n✅ Elec2 dataset cleaned successfully (temporal order preserved)")

In [ ]:
# ── Save cleaned elec2 ──
elec2_clean.to_csv('elec2_clean.csv', index=False)
print(f"Saved: elec2_clean.csv ({elec2_clean.shape[0]:,} rows, {elec2_clean.shape[1]} cols)")

---
## 3. LendingClub Dataset

**Source:** [Kaggle — LendingClub Loan Data](https://www.kaggle.com/datasets/wordsforthewise/lending-club)

**Raw:** ~2.2M rows, 151 columns (mix of text, dates, numerics, sparse joint-app fields)

**Cleaning steps:**
1. Filter to terminal statuses only: `Fully Paid` and `Charged Off`
2. Create target: `Charged Off` = 1, `Fully Paid` = 0
3. Parse `issue_d` for chronological ordering
4. Select **18 curated numeric features** (drop IDs, text, dates, post-outcome, sparse)
5. Parse string-encoded numerics:
   - `term`: `" 36 months"` → `36`
   - `int_rate`: `"10.65%"` → `10.65`  
   - `revol_util`: `"50.5%"` → `50.5`
6. Convert all columns to numeric
7. Impute NaN with column median
8. Sort by `issue_d` (chronological order for drift)

In [ ]:
# ── Upload or load the Kaggle CSV ──
# The Kaggle dataset is large (~1.6 GB)
# On Colab: either upload or use Kaggle API

try:
    from google.colab import files
    print("Upload the LendingClub CSV (accepted_2007_to_2018Q4.csv or similar):")
    uploaded = files.upload()
    lc_path = list(uploaded.keys())[0]
except ImportError:
    lc_path = r"C:\claude\lendingclub_clean.csv"  # Change this

print(f"Reading from: {lc_path}")

In [ ]:
# ── Load raw data ──
print("Loading LendingClub CSV (this may take a minute)...")
lc_raw = pd.read_csv(lc_path, low_memory=False)

print("\n" + "=" * 60)
print("LENDINGCLUB — RAW DATA")
print("=" * 60)
print(f"Shape: {lc_raw.shape}")
print(f"Columns ({len(lc_raw.columns)}): {list(lc_raw.columns[:10])}...")
print(f"\nDtype counts:\n{lc_raw.dtypes.value_counts()}")
print(f"\nTotal NaN cells: {lc_raw.isna().sum().sum():,}")

# Show loan_status distribution
if 'loan_status' in lc_raw.columns:
    print(f"\nloan_status distribution:")
    print(lc_raw['loan_status'].value_counts())

In [ ]:
# ── Step 1: Filter to terminal statuses ──
print("Step 1: Filtering to Fully Paid / Charged Off only...")

valid_statuses = ['Fully Paid', 'Charged Off']
mask = lc_raw['loan_status'].isin(valid_statuses)
lc_filtered = lc_raw[mask].copy()
print(f"  Before: {len(lc_raw):,} rows")
print(f"  After:  {len(lc_filtered):,} rows")
print(f"  Dropped: {len(lc_raw) - len(lc_filtered):,} rows (Current, Late, etc.)")

In [ ]:
# ── Step 2: Create binary target ──
print("Step 2: Creating binary target (Charged Off = 1, Fully Paid = 0)...")

lc_filtered['target'] = (lc_filtered['loan_status'] == 'Charged Off').astype(int)
print(f"  Target distribution:")
print(f"    Fully Paid (0):  {(lc_filtered['target']==0).sum():,}")
print(f"    Charged Off (1): {(lc_filtered['target']==1).sum():,}")
print(f"    Positive rate:   {lc_filtered['target'].mean():.4f}")

In [ ]:
# ── Step 3: Parse issue_d and sort chronologically ──
print("Step 3: Parsing issue_d for chronological ordering...")

lc_filtered['issue_date_parsed'] = pd.to_datetime(
    lc_filtered['issue_d'], format='mixed', errors='coerce'
)

n_valid_dates = lc_filtered['issue_date_parsed'].notna().sum()
print(f"  Valid dates: {n_valid_dates:,} / {len(lc_filtered):,}")
print(f"  Date range: {lc_filtered['issue_date_parsed'].min()} to {lc_filtered['issue_date_parsed'].max()}")

lc_filtered = lc_filtered.sort_values('issue_date_parsed').reset_index(drop=True)
print("  Sorted by issue_d ✓")

In [ ]:
# ── Step 4: Select 18 curated numeric features ──
print("Step 4: Selecting 18 curated features...")

SELECTED_FEATURES = [
    'loan_amnt',         # Loan amount requested
    'funded_amnt',       # Amount funded by investors
    'term',              # 36 or 60 months (needs parsing)
    'int_rate',          # Interest rate (needs parsing)
    'installment',       # Monthly payment amount
    'annual_inc',        # Annual income
    'dti',               # Debt-to-income ratio
    'delinq_2yrs',       # Delinquencies in past 2 years
    'fico_range_low',    # FICO score lower bound
    'fico_range_high',   # FICO score upper bound
    'inq_last_6mths',    # Credit inquiries in past 6 months
    'open_acc',          # Open credit lines
    'pub_rec',           # Public records (bankruptcies etc)
    'revol_bal',         # Revolving balance
    'revol_util',        # Revolving utilization rate (needs parsing)
    'total_acc',         # Total credit lines
    'acc_now_delinq',    # Accounts currently delinquent
    'tot_coll_amt',      # Total collection amounts
]

# Check which features exist
available = [c for c in SELECTED_FEATURES if c in lc_filtered.columns]
missing = [c for c in SELECTED_FEATURES if c not in lc_filtered.columns]
print(f"  Available: {len(available)}/18 features")
if missing:
    print(f"  Missing: {missing}")

print(f"  Dropping {len(lc_filtered.columns) - len(available) - 1} unused columns")
print(f"  Kept: {available}")

In [ ]:
# ── Step 5: Parse string-encoded numerics ──
print("Step 5: Parsing string-encoded numerics...")

lc_clean = lc_filtered[available].copy()

# term: " 36 months" → 36
if 'term' in lc_clean.columns:
    before = lc_clean['term'].head(3).tolist()
    lc_clean['term'] = (
        lc_clean['term'].astype(str)
        .str.replace('months', '', regex=False)
        .str.strip()
    )
    lc_clean['term'] = pd.to_numeric(lc_clean['term'], errors='coerce')
    after = lc_clean['term'].head(3).tolist()
    print(f"  term: {before} → {after}")

# int_rate: "10.65%" → 10.65
if 'int_rate' in lc_clean.columns:
    before = lc_clean['int_rate'].head(3).tolist()
    lc_clean['int_rate'] = (
        lc_clean['int_rate'].astype(str)
        .str.replace('%', '', regex=False)
        .str.strip()
    )
    lc_clean['int_rate'] = pd.to_numeric(lc_clean['int_rate'], errors='coerce')
    after = lc_clean['int_rate'].head(3).tolist()
    print(f"  int_rate: {before} → {after}")

# revol_util: "50.5%" → 50.5
if 'revol_util' in lc_clean.columns:
    before = lc_clean['revol_util'].head(3).tolist()
    lc_clean['revol_util'] = (
        lc_clean['revol_util'].astype(str)
        .str.replace('%', '', regex=False)
        .str.strip()
    )
    lc_clean['revol_util'] = pd.to_numeric(lc_clean['revol_util'], errors='coerce')
    after = lc_clean['revol_util'].head(3).tolist()
    print(f"  revol_util: {before} → {after}")

In [ ]:
# ── Step 6: Convert all to numeric ──
print("Step 6: Converting all columns to numeric...")

for col in lc_clean.columns:
    lc_clean[col] = pd.to_numeric(lc_clean[col], errors='coerce')

print(f"  Dtypes: {lc_clean.dtypes.value_counts().to_dict()}")

In [ ]:
# ── Step 7: Handle NaN ──
print("Step 7: Handling NaN values...")

na_counts = lc_clean.isna().sum()
na_cols = na_counts[na_counts > 0]
print(f"  Columns with NaN ({len(na_cols)}):")
for col, count in na_cols.items():
    pct = count / len(lc_clean) * 100
    print(f"    {col:25s}: {count:>10,} ({pct:.2f}%)")

# Impute with median
print("\n  Imputing with column median...")
lc_clean = lc_clean.fillna(lc_clean.median())

# Drop any remaining NaN rows (edge case)
n_before = len(lc_clean)
lc_clean = lc_clean.dropna()
n_dropped = n_before - len(lc_clean)
if n_dropped > 0:
    print(f"  Dropped {n_dropped} rows with remaining NaN")

print(f"  Final NaN count: {lc_clean.isna().sum().sum()}")

In [ ]:
# ── Add target column ──
lc_clean['target'] = lc_filtered.loc[lc_clean.index, 'target'].values

# ── Verify ──
print("\n" + "=" * 60)
print("LENDINGCLUB — CLEANED")
print("=" * 60)
print(f"Shape: {lc_clean.shape}")
feature_cols_lc = [c for c in lc_clean.columns if c != 'target']
print(f"Features ({len(feature_cols_lc)}): {feature_cols_lc}")
print(f"NaN cells: {lc_clean.isna().sum().sum()}")
print(f"Target: {{0: {(lc_clean['target']==0).sum():,}, 1: {(lc_clean['target']==1).sum():,}}}")
print(f"Positive rate: {lc_clean['target'].mean():.4f} ({lc_clean['target'].mean()*100:.2f}%)")
print(f"All dtypes numeric: {all(lc_clean.dtypes.apply(lambda x: np.issubdtype(x, np.number)))}")
print(f"\nFirst 5 rows:")
lc_clean.head()

In [ ]:
# ── Save cleaned lendingclub ──
lc_clean.to_csv('lendingclub_clean.csv', index=False)
print(f"Saved: lendingclub_clean.csv ({lc_clean.shape[0]:,} rows, {lc_clean.shape[1]} cols)")

---
## 4. Verification Summary

In [ ]:
# ── Final summary of all 3 cleaned datasets ──
print("=" * 70)
print("FINAL SUMMARY — ALL CLEANED DATASETS")
print("=" * 70)
print(f"{'Dataset':<18s} {'Rows':>10s} {'Features':>10s} {'Pos Rate':>10s} {'NaN':>6s} {'Status'}")
print("-" * 70)

datasets = [
    ('Fraud',       fraud_clean, [c for c in fraud_clean.columns if c != 'target']),
    ('Elec2',       elec2_clean, [c for c in elec2_clean.columns if c != 'target']),
    ('LendingClub', lc_clean,    [c for c in lc_clean.columns if c != 'target']),
]

for name, df, feats in datasets:
    n_rows = len(df)
    n_feats = len(feats)
    pos_rate = df['target'].mean()
    n_nan = df.isna().sum().sum()
    all_numeric = all(df[feats].dtypes.apply(lambda x: np.issubdtype(x, np.number)))
    status = '✅' if n_nan == 0 and all_numeric else '❌'
    print(f"{name:<18s} {n_rows:>10,} {n_feats:>10} {pos_rate:>10.4f} {n_nan:>6} {status}")

print("\n" + "=" * 70)
print("Files saved:")
print("  1. fraud_clean.csv")
print("  2. elec2_clean.csv")
print("  3. lendingclub_clean.csv")
print("\n✅ All datasets cleaned and ready for the harness!")

---
## 5. Download Files (Colab Only)

Run this cell on Google Colab to download the cleaned CSVs to your local machine.

In [ ]:
# ── Download from Colab ──
try:
    from google.colab import files
    print("Downloading cleaned CSVs...")
    files.download('fraud_clean.csv')
    files.download('elec2_clean.csv')
    files.download('lendingclub_clean.csv')
except ImportError:
    print("Not on Colab — files already saved locally.")

---
## 6. Drift Detection on the Cleaned Datasets (Part 1 Verification)

Now that the datasets are cleaned, we can run our **Part 1 Drift Detectors** on them:
- **Evidently-Style**: Two-sample Kolmogorov-Smirnov (KS) test + Population Stability Index (PSI) per feature, with a majority vote (>50% drifted features).
- **Alibi-Style (BH-FDR)**: Feature-level KS tests with Benjamini-Hochberg False Discovery Rate correction.
- **Alibi-Style (MMD)**: Multivariate Maximum Mean Discrepancy using an RBF kernel with permutation testing for joint distribution shift.

In [ ]:
# ── Clone or import the Part 1 detectors ──
import sys
import os
# If running locally or from git repo, add comparison-study to path:
for p in ['.', '..', 'comparison-study', '/content/self-healing-mlops/comparison-study']:
    if os.path.exists(p):
        sys.path.insert(0, os.path.abspath(p))

try:
    from part1_detect.evidently_style_detector import EvidentlyStyleDriftDetector
    from part1_detect.alibi_style_detector import AlibiStyleDriftDetector
    print("✅ Part 1 detectors successfully imported!")
except ImportError:
    print("⚠️ Could not find part1_detect in python path. Make sure the repository is cloned or files are in the same folder.")

### 6.1 Verification on Fraud Dataset

We test two scenarios:
1. **Stationary / Clean**: Reference (1,000 samples) vs Current (500 samples) drawn from the same training distribution (expected: **NO DRIFT**).
2. **Injected Drift**: Current has synthetic +2σ mean shift applied to top 5 features (expected: **DRIFT DETECTED**).

In [ ]:
# ── Test Fraud Dataset ──
feature_cols_fraud = [c for c in fraud_clean.columns if c != 'target']

# Sample 1000 reference rows and 500 current rows
X_ref_fraud = fraud_clean[feature_cols_fraud].sample(n=1000, random_state=42)
X_cur_clean = fraud_clean[feature_cols_fraud].sample(n=500, random_state=99)

# Injected drift (+2 sigma shift on V1..V5)
X_cur_drift = X_cur_clean.copy()
for col in ['V1', 'V2', 'V3', 'V4', 'V5']:
    X_cur_drift[col] = X_cur_drift[col] + 2.0 * X_cur_drift[col].std()

ref_dicts = X_ref_fraud.to_dict(orient='records')
clean_dicts = X_cur_clean.to_dict(orient='records')
drift_dicts = X_cur_drift.to_dict(orient='records')

ev_det = EvidentlyStyleDriftDetector(ks_alpha=0.05, psi_threshold=0.20)
alibi_bh = AlibiStyleDriftDetector(method='columnwise_bh', alpha=0.05)
alibi_mmd = AlibiStyleDriftDetector(method='multivariate_mmd', alpha=0.05, n_permutations=100, seed=42)

print("="*70)
print("FRAUD SCENARIO 1: Clean vs Clean (Expect: No Drift)")
print("="*70)
res_clean_ev = ev_det.detect(ref_dicts, clean_dicts, feature_cols_fraud, [])
res_clean_bh = alibi_bh.detect(ref_dicts, clean_dicts, feature_cols_fraud, [])
res_clean_mmd = alibi_mmd.detect(ref_dicts[:80], clean_dicts[:80], feature_cols_fraud, [])

print(f"Evidently-style: drift={res_clean_ev['dataset_drift_detected']} ({res_clean_ev['n_features_drifted']}/29 features drifted)")
print(f"Alibi-BH:        drift={res_clean_bh['dataset_drift_detected']} ({res_clean_bh['n_features_drifted']}/29 features rejected)")
print(f"Alibi-MMD:       drift={res_clean_mmd['dataset_drift_detected']} (p={res_clean_mmd['p_value']:.4f})")

print("\n" + "="*70)
print("FRAUD SCENARIO 2: Injected Drift on V1-V5 (Expect: Drift)")
print("="*70)
res_drift_ev = ev_det.detect(ref_dicts, drift_dicts, feature_cols_fraud, [])
res_drift_bh = alibi_bh.detect(ref_dicts, drift_dicts, feature_cols_fraud, [])
res_drift_mmd = alibi_mmd.detect(ref_dicts[:80], drift_dicts[:80], feature_cols_fraud, [])

print(f"Evidently-style: drift={res_drift_ev['dataset_drift_detected']} ({res_drift_ev['n_features_drifted']}/29 features drifted)")
print(f"Alibi-BH:        drift={res_drift_bh['dataset_drift_detected']} ({res_drift_bh['n_features_drifted']}/29 features rejected)")
print(f"Alibi-MMD:       drift={res_drift_mmd['dataset_drift_detected']} (p={res_drift_mmd['p_value']:.4f})")

### 6.2 Verification on LendingClub (Real Temporal Drift Across Years)

Here we compare:
- **Reference**: Early loans from the start of the timeline (first 1,000 rows).
- **Current**: Later loans from the middle and end of the timeline.

In [ ]:
# ── Test LendingClub Timeline ──
feature_cols_lc = [c for c in lc_clean.columns if c != 'target']

ref_lc = lc_clean[feature_cols_lc].iloc[:1000].to_dict(orient='records')
mid_lc = lc_clean[feature_cols_lc].iloc[len(lc_clean)//2 : len(lc_clean)//2 + 500].to_dict(orient='records')
late_lc = lc_clean[feature_cols_lc].iloc[-500:].to_dict(orient='records')

print("="*70)
print("LENDINGCLUB: Early Reference vs Middle Timeline")
print("="*70)
mid_ev = ev_det.detect(ref_lc, mid_lc, feature_cols_lc, [])
mid_bh = alibi_bh.detect(ref_lc, mid_lc, feature_cols_lc, [])
mid_mmd = alibi_mmd.detect(ref_lc[:80], mid_lc[:80], feature_cols_lc, [])
print(f"Evidently: drift={mid_ev['dataset_drift_detected']} ({mid_ev['n_features_drifted']}/18 drifted)")
print(f"Alibi-BH:  drift={mid_bh['dataset_drift_detected']} ({mid_bh['n_features_drifted']}/18 rejected)")
print(f"Alibi-MMD: drift={mid_mmd['dataset_drift_detected']} (p={mid_mmd['p_value']:.4f})")

print("\n" + "="*70)
print("LENDINGCLUB: Early Reference vs Late Timeline")
print("="*70)
late_ev = ev_det.detect(ref_lc, late_lc, feature_cols_lc, [])
late_bh = alibi_bh.detect(ref_lc, late_lc, feature_cols_lc, [])
late_mmd = alibi_mmd.detect(ref_lc[:80], late_lc[:80], feature_cols_lc, [])
print(f"Evidently: drift={late_ev['dataset_drift_detected']} ({late_ev['n_features_drifted']}/18 drifted)")
print(f"Alibi-BH:  drift={late_bh['dataset_drift_detected']} ({late_bh['n_features_drifted']}/18 rejected)")
print(f"Alibi-MMD: drift={late_mmd['dataset_drift_detected']} (p={late_mmd['p_value']:.4f})")